# 1. DATA CLEANING
## Daily Customer Churn Predictor · VivaMarket Brasil

---

**INPUT:** `../data/raw/churn_sqlite_db.sqlite`  
*An anonymized ecommerce SQLite database with transactional, customer, product, payment, review and logistics information for VivaMarket Brasil.*

**OUTPUT:** `../data/interim/client_database_clean_YYYYMMDD.parquet`  
*A cleaned and customer-centered analytical base ready for exploratory analysis and churn feature engineering.*


---
## 1.1. STARTING SITUATION


VivaMarket Brasil is a large ecommerce marketplace that needs to **predict customer churn daily for the next 90 days** and activate retention actions before high-value customers disengage.

The raw data is stored in a relational SQLite database rather than in a modeling-ready flat table. This means the first notebook must transform a multi-table operational source into a **consistent analytical dataset** centered on customer behavior.

At this stage, the main risks are not yet model-related. The most immediate risks are:

- inconsistent joins between orders, customers, items, payments and reviews;
- missing or partially populated operational timestamps;
- duplicated customer identities caused by marketplace ordering behavior;
- leakage risks if post-outcome information is accidentally used in future labeling or prediction logic.


### 1.1.1. DATASET ORIGIN

The dataset source documented in the project points to the public Olist ecommerce SQLite adaptation hosted on Kaggle. For this project, the database is used as an anonymized proxy for VivaMarket Brasil business data and respects the project privacy framing.


---
## 1.2. MODEL OBJECTIVE


- **Business objective:** identify customers with high probability of churn in the next 90 days so the business can intervene before abandonment.
- **Analytical objective:** build a reproducible customer-level base that supports temporal churn labeling, behavioral feature engineering and production scoring.
- **Notebook objective:** clean, validate and structure the raw relational data so downstream notebooks can work on a stable customer-centric foundation instead of raw operational tables.

This notebook is intentionally focused on the **what** and the **why** of data preparation. The result we need here is not a finished model but a trustworthy dataset contract for the rest of the project.


---
## 1.3. INITIAL SETUP


**What is done**

We import the libraries required to connect to SQLite, manipulate tabular data, log the notebook execution and manage project-relative paths.

**Why it is done**

The project must be reproducible across environments, so path handling is based on the project structure rather than on machine-specific absolute paths. Logging is included from the start to make each execution step auditable and easier to debug.

**Expected result**

A stable execution context that can load the raw database using project-relative paths and report major steps through `INFO` logs.


In [1]:
import logging
from pathlib import Path
import sqlite3
from datetime import datetime

import pandas as pd
import numpy as np

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)
logger.info('NB01 started: data cleaning and initial preparation.')

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


2026-04-29 16:34:20,213 | INFO | NB01 started: data cleaning and initial preparation.


In [2]:
def resolve_project_root() -> Path:
    candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for candidate in candidate_roots:
        if (candidate / 'data' / 'raw' / 'churn_sqlite_db.sqlite').exists():
            return candidate
        if (candidate / 'daily-customer-churn-predictor').exists() and (candidate / 'daily-customer-churn-predictor' / 'data' / 'raw' / 'churn_sqlite_db.sqlite').exists():
            return candidate / 'daily-customer-churn-predictor'
    raise FileNotFoundError('Could not resolve project root containing data/raw/churn_sqlite_db.sqlite')

PROJECT_ROOT = resolve_project_root()
PATH_DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
PATH_DATA_INTERIM = PROJECT_ROOT / 'data' / 'interim'
PATH_NOTEBOOKS = PROJECT_ROOT / 'notebooks'
DB_PATH = PATH_DATA_RAW / 'churn_sqlite_db.sqlite'

PATH_DATA_INTERIM.mkdir(parents=True, exist_ok=True)

logger.info('Project root resolved at %s', PROJECT_ROOT)
logger.info('SQLite source path resolved at %s', DB_PATH)
assert DB_PATH.exists(), f'Missing database at {DB_PATH}'


2026-04-29 16:34:20,220 | INFO | Project root resolved at /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor


2026-04-29 16:34:20,220 | INFO | SQLite source path resolved at /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/raw/churn_sqlite_db.sqlite


In [3]:
try:
    conn = sqlite3.connect(DB_PATH)
    logger.info('SQLite connection established successfully.')
except Exception as exc:
    logger.error('Failed to connect to SQLite database: %s', exc)
    raise


2026-04-29 16:34:20,224 | INFO | SQLite connection established successfully.


---
## 1.4. PRELIMINARY DATASET ANALYSIS


**What is done**

We inspect the raw relational source to understand table inventory, row volumes, timestamp coverage, order statuses and customer identity behavior.

**Why it is done**

Before cleaning anything, we need to understand the operational structure of the source. This prevents wrong assumptions about analytical grain, customer uniqueness and missing-value behavior.

**Expected result**

A concise diagnostic view of the database showing whether it is structurally usable for churn modeling.


In [4]:
table_inventory_query = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
"""

tables = pd.read_sql_query(table_inventory_query, conn)
logger.info('Discovered %d tables in the SQLite source.', len(tables))
tables


2026-04-29 16:34:20,232 | INFO | Discovered 11 tables in the SQLite source.


,name
0,customers
1,geolocation
2,leads_closed
3,leads_qualified
4,order_items
5,order_payments
6,order_reviews
7,orders
8,product_category_name_translation
9,products


In [5]:
table_counts = []
for table_name in tables["name"]:
    query = f"SELECT COUNT(*) AS row_count FROM {table_name}"
    row_count = pd.read_sql_query(query, conn).iloc[0, 0]
    table_counts.append({"table_name": table_name, "row_count": row_count})

table_counts_df = pd.DataFrame(table_counts).sort_values("row_count", ascending=False)
table_counts_df


,table_name,row_count
1,geolocation,1000163
4,order_items,112650
5,order_payments,103886
7,orders,99441
0,customers,99441
6,order_reviews,99224
9,products,32951
3,leads_qualified,8000
10,sellers,3095
2,leads_closed,842


In [6]:
orders_overview_query = """
SELECT
    MIN(order_purchase_timestamp) AS min_purchase_timestamp,
    MAX(order_purchase_timestamp) AS max_purchase_timestamp,
    COUNT(DISTINCT order_id) AS distinct_orders,
    COUNT(DISTINCT customer_id) AS distinct_order_level_customers
FROM orders
"""

orders_overview = pd.read_sql_query(orders_overview_query, conn)
orders_overview


,min_purchase_timestamp,max_purchase_timestamp,distinct_orders,distinct_order_level_customers
0,2016-09-04 21:15:19,2018-10-17 17:30:18,99441,99441


In [7]:
orders_status_query = """
SELECT order_status, COUNT(*) AS orders_n
FROM orders
GROUP BY order_status
ORDER BY orders_n DESC
"""

orders_status = pd.read_sql_query(orders_status_query, conn)
orders_status


,order_status,orders_n
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [8]:
missing_operational_dates_query = """
SELECT
    SUM(order_purchase_timestamp IS NULL) AS purchase_timestamp_nulls,
    SUM(order_approved_at IS NULL) AS approved_at_nulls,
    SUM(order_delivered_carrier_date IS NULL) AS delivered_carrier_nulls,
    SUM(order_delivered_customer_date IS NULL) AS delivered_customer_nulls,
    SUM(order_estimated_delivery_date IS NULL) AS estimated_delivery_nulls
FROM orders
"""

missing_operational_dates = pd.read_sql_query(missing_operational_dates_query, conn)
missing_operational_dates


,purchase_timestamp_nulls,approved_at_nulls,delivered_carrier_nulls,delivered_customer_nulls,estimated_delivery_nulls
0,0,160,1783,2965,0


In [9]:
customer_identity_query = """
SELECT
    COUNT(*) AS customer_rows,
    COUNT(DISTINCT customer_id) AS distinct_customer_id,
    COUNT(DISTINCT customer_unique_id) AS distinct_customer_unique_id
FROM customers
"""

customer_identity = pd.read_sql_query(customer_identity_query, conn)
customer_identity


,customer_rows,distinct_customer_id,distinct_customer_unique_id
0,99441,99441,96096


In [10]:
payments_summary_query = """
SELECT
    payment_type,
    COUNT(*) AS payments_n,
    ROUND(AVG(payment_value), 2) AS avg_payment_value
FROM order_payments
GROUP BY payment_type
ORDER BY payments_n DESC
"""

payments_summary = pd.read_sql_query(payments_summary_query, conn)
payments_summary


,payment_type,payments_n,avg_payment_value
0,credit_card,76795,163.32
1,boleto,19784,145.03
2,voucher,5775,65.70
3,debit_card,1529,142.57
4,not_defined,3,0.00


In [11]:
review_summary_query = """
SELECT
    MIN(review_score) AS min_review_score,
    MAX(review_score) AS max_review_score,
    ROUND(AVG(review_score), 4) AS avg_review_score
FROM order_reviews
"""

review_summary = pd.read_sql_query(review_summary_query, conn)
review_summary


,min_review_score,max_review_score,avg_review_score
0,1,5,4.09


In [12]:
order_level_join_query = """
SELECT
    COUNT(*) AS order_rows,
    COUNT(DISTINCT o.order_id) AS distinct_orders,
    COUNT(DISTINCT c.customer_unique_id) AS distinct_customer_unique_id,
    ROUND(AVG(oi.item_count), 2) AS avg_items_per_order,
    ROUND(AVG(op.payment_total), 2) AS avg_payment_per_order,
    ROUND(AVG(orv.review_score), 2) AS avg_review_score
FROM orders o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id
LEFT JOIN (
    SELECT order_id, COUNT(*) AS item_count
    FROM order_items
    GROUP BY order_id
) oi
    ON o.order_id = oi.order_id
LEFT JOIN (
    SELECT order_id, SUM(payment_value) AS payment_total
    FROM order_payments
    GROUP BY order_id
) op
    ON o.order_id = op.order_id
LEFT JOIN (
    SELECT order_id, AVG(review_score) AS review_score
    FROM order_reviews
    GROUP BY order_id
) orv
    ON o.order_id = orv.order_id
"""

order_level_join_summary = pd.read_sql_query(order_level_join_query, conn)
order_level_join_summary


,order_rows,distinct_orders,distinct_customer_unique_id,avg_items_per_order,avg_payment_per_order,avg_review_score
0,99441,99441,96096,1.14,160.99,4.09


---
## 1.5. DATA LOADING


**What is done**

We load the core operational tables required for churn preparation: `orders`, `customers`, `order_items`, `order_payments`, `order_reviews` and `products`.

**Why it is done**

These tables contain the minimum information needed to reconstruct purchasing behavior, monetary value, satisfaction signals and product/category context at customer level.

**Expected result**

A local in-memory working set that supports deterministic joins and downstream cleaning steps.


In [13]:
orders_df = pd.read_sql_query("SELECT * FROM orders", conn)
customers_df = pd.read_sql_query("SELECT * FROM customers", conn)
order_items_df = pd.read_sql_query("SELECT * FROM order_items", conn)
order_payments_df = pd.read_sql_query("SELECT * FROM order_payments", conn)
order_reviews_df = pd.read_sql_query("SELECT * FROM order_reviews", conn)
products_df = pd.read_sql_query("SELECT * FROM products", conn)

logger.info('Core tables loaded into pandas DataFrames.')
pd.DataFrame({
    "table_name": ["orders", "customers", "order_items", "order_payments", "order_reviews", "products"],
    "rows": [len(orders_df), len(customers_df), len(order_items_df), len(order_payments_df), len(order_reviews_df), len(products_df)],
    "columns": [orders_df.shape[1], customers_df.shape[1], order_items_df.shape[1], order_payments_df.shape[1], order_reviews_df.shape[1], products_df.shape[1]]
})


2026-04-29 16:34:24,611 | INFO | Core tables loaded into pandas DataFrames.


,table_name,rows,columns
0,orders,99441,8
1,customers,99441,5
2,order_items,112650,7
3,order_payments,103886,5
4,order_reviews,99224,7
5,products,32951,9


---
## 1.6. DATA CLEANING


**What is done**

We standardize timestamps, aggregate one-to-many order tables, align customer identifiers and build a first customer-level analytical dataset.

**Why it is done**

Churn is a customer-level outcome. The raw source is order-level and fragmented across several related tables. We must therefore convert operational granularity into a stable customer-centric representation before any churn labeling or feature engineering.

**Expected result**

A clean dataframe where each row represents one customer unique identifier enriched with purchasing, payment, review and product-category summary behavior.


In [14]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    try:
        orders_df[column] = pd.to_datetime(orders_df[column], errors="coerce")
    except Exception as exc:
        logger.error("Failed to convert %s to datetime: %s", column, exc)
        raise

logger.info('Order timestamps converted to datetime.')


2026-04-29 16:34:24,694 | INFO | Order timestamps converted to datetime.


In [15]:
order_items_agg = (
    order_items_df.groupby("order_id", as_index=False)
    .agg(
        items_n=("order_item_id", "count"),
        total_item_price=("price", "sum"),
        total_freight_value=("freight_value", "sum"),
        unique_products_n=("product_id", "nunique"),
        unique_sellers_n=("seller_id", "nunique")
    )
)

order_payments_agg = (
    order_payments_df.groupby("order_id", as_index=False)
    .agg(
        payment_records_n=("payment_sequential", "count"),
        total_payment_value=("payment_value", "sum"),
        max_installments=("payment_installments", "max")
    )
)

order_reviews_agg = (
    order_reviews_df.groupby("order_id", as_index=False)
    .agg(
        avg_review_score=("review_score", "mean"),
        review_records_n=("review_id", "count")
    )
)

logger.info('Order-level one-to-many tables aggregated successfully.')


2026-04-29 16:34:24,823 | INFO | Order-level one-to-many tables aggregated successfully.


In [16]:
orders_enriched_df = (
    orders_df
    .merge(customers_df, on="customer_id", how="left")
    .merge(order_items_agg, on="order_id", how="left")
    .merge(order_payments_agg, on="order_id", how="left")
    .merge(order_reviews_agg, on="order_id", how="left")
)

orders_enriched_df["delivered_flag"] = (orders_enriched_df["order_status"] == "delivered").astype(int)
orders_enriched_df["purchase_year"] = orders_enriched_df["order_purchase_timestamp"].dt.year
orders_enriched_df["purchase_month"] = orders_enriched_df["order_purchase_timestamp"].dt.month

logger.info('Orders table enriched with customer, item, payment and review summaries.')


2026-04-29 16:34:24,967 | INFO | Orders table enriched with customer, item, payment and review summaries.


In [17]:
customer_level_df = (
    orders_enriched_df.groupby("customer_unique_id", as_index=False)
    .agg(
        customer_state=("customer_state", "last"),
        first_purchase_timestamp=("order_purchase_timestamp", "min"),
        last_purchase_timestamp=("order_purchase_timestamp", "max"),
        orders_n=("order_id", "nunique"),
        delivered_orders_n=("delivered_flag", "sum"),
        total_items_n=("items_n", "sum"),
        total_payment_value=("total_payment_value", "sum"),
        total_item_price=("total_item_price", "sum"),
        total_freight_value=("total_freight_value", "sum"),
        avg_review_score=("avg_review_score", "mean"),
        max_installments=("max_installments", "max")
    )
)

customer_level_df["customer_lifetime_days"] = (
    customer_level_df["last_purchase_timestamp"] - customer_level_df["first_purchase_timestamp"]
).dt.days

customer_level_df["avg_order_value"] = customer_level_df["total_payment_value"] / customer_level_df["orders_n"]
customer_level_df["avg_items_per_order"] = customer_level_df["total_items_n"] / customer_level_df["orders_n"]

logger.info('Customer-level analytical base created with %d rows.', len(customer_level_df))
customer_level_df.head()


2026-04-29 16:34:25,066 | INFO | Customer-level analytical base created with 96096 rows.


,customer_unique_id,customer_state,first_purchase_timestamp,last_purchase_timestamp,orders_n,delivered_orders_n,total_items_n,total_payment_value,total_item_price,total_freight_value,avg_review_score,max_installments,customer_lifetime_days,avg_order_value,avg_items_per_order
0,0000366f3b9a7992bf8c76cfdf3221e2,SP,2018-05-10 10:56:27,2018-05-10 10:56:27,1,1,1.00,141.90,129.90,12.00,5.00,8.00,0,141.90,1.00
1,0000b849f77a49e4a4ce2b2a4ca5be3f,SP,2018-05-07 11:11:27,2018-05-07 11:11:27,1,1,1.00,27.19,18.90,8.29,4.00,1.00,0,27.19,1.00
2,0000f46a3911fa3c0805444483337064,SC,2017-03-10 21:05:03,2017-03-10 21:05:03,1,1,1.00,86.22,69.00,17.22,3.00,8.00,0,86.22,1.00
3,0000f6ccb0745a6a4b88665a16c9f078,PA,2017-10-12 20:29:41,2017-10-12 20:29:41,1,1,1.00,43.62,25.99,17.63,4.00,4.00,0,43.62,1.00
4,0004aac84e0df4da2b147fca70cf8255,SP,2017-11-14 19:45:42,2017-11-14 19:45:42,1,1,1.00,196.89,180.00,16.89,5.00,6.00,0,196.89,1.00


---
## 1.7. INITIAL OUTPUT GENERATION


**What is done**

We export the cleaned customer-level base as the initial interim deliverable for the rest of the project.

**Why it is done**

The following notebooks should not repeatedly rebuild the full operational extraction unless necessary. Saving a first analytical base improves reproducibility and reduces friction for later stages.

**Expected result**

A dated interim file stored inside `data/interim/` and ready to support EDA and churn target definition.


In [18]:
run_date = datetime.now().strftime("%Y%m%d")
output_path = PATH_DATA_INTERIM / f"client_database_clean_{run_date}.parquet"

try:
    customer_level_df.to_parquet(output_path, index=False)
    logger.info("Customer-level interim dataset exported to %s", output_path)
except Exception as exc:
    logger.error("Failed to export parquet output: %s", exc)
    raise

output_path


2026-04-29 16:34:25,151 | INFO | Customer-level interim dataset exported to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/interim/client_database_clean_20260429.parquet


PosixPath('/data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/interim/client_database_clean_20260429.parquet')

---
## 1.8. RESULTS AND NEXT STEP


**Result of this notebook**

The raw SQLite source is structurally valid for churn modeling and can already be transformed into a first customer-level analytical table. The source shows strong delivered-order dominance, manageable missingness in operational dates, and a meaningful distinction between `customer_id` and `customer_unique_id`, which will be critical for future churn labeling.

**Why this matters**

This notebook establishes the data contract for the rest of the project. We now know the correct customer key, the key operational tables and the baseline aggregation logic required for churn work.

**Next step**

The next notebook should perform a deep exploratory analysis on the cleaned customer base, formalize the churn definition for a 90-day horizon and identify the most relevant segmentation and temporal patterns for feature engineering.
